In [ ]:
import tensorflow.keras as keras
import tensorflow.keras.layers as layers

model = keras.Sequential()
model.add(layers.Conv2D(filters=6, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1)))
model.add(layers.AveragePooling2D())
model.add(layers.Conv2D(filters=16, kernel_size=(3, 3), activation='relu'))
model.add(layers.AveragePooling2D())
model.add(layers.Flatten())
model.add(layers.Dense(units=120, activation='relu'))
model.add(layers.Dense(units=84, activation='relu'))
model.add(layers.Dense(units=10, activation = 'softmax'))

In [ ]:
model

In [ ]:
model.summary()

In [ ]:
from keras.datasets import mnist
from keras import backend as K

In [ ]:
batch_size = 128
num_classes = 10
epochs = 12

# input image dimensions
img_rows, img_cols = 28, 28

# the data, split between train and test sets
(x_train, y_train), (x_test, y_test) = mnist.load_data()

if K.image_data_format() == 'channels_first':
    x_train = x_train.reshape(x_train.shape[0], 1, img_rows, img_cols)
    x_test = x_test.reshape(x_test.shape[0], 1, img_rows, img_cols)
    input_shape = (1, img_rows, img_cols)
else:
    x_train = x_train.reshape(x_train.shape[0], img_rows, img_cols, 1)
    x_test = x_test.reshape(x_test.shape[0], img_rows, img_cols, 1)
    input_shape = (img_rows, img_cols, 1)

In [ ]:
x_train.shape

In [ ]:
y_train.shape

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
%matplotlib inline
plt.imshow(x_train[0][:, :, 0], cmap=cm.get_cmap("Greys"))
plt.title(y_train[0])

In [ ]:
model.compile(loss=keras.losses.categorical_crossentropy, optimizer=keras.optimizers.Adam(), metrics=['accuracy'])

In [ ]:
EPOCHS = 10
BATCH_SIZE = 128

In [ ]:
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255

In [ ]:
num_classes = 10
y_train = keras.utils.to_categorical(y_train, num_classes)
y_test = keras.utils.to_categorical(y_test, num_classes)

In [ ]:
y_train[0]

In [ ]:
model.fit(x_train, y_train,
          batch_size=batch_size,
          epochs=epochs,
          verbose=1,
          validation_data=(x_test, y_test))
score = model.evaluate(x_test, y_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

In [ ]:
plt.imshow(model.get_weights()[0][:, :, 0, 0])

In [ ]:
plt.imshow(model.get_weights()[0][:, :, 0, 1])

In [ ]:
len(model.get_weights())

In [ ]:
e = model.layers[0]

In [ ]:
e.get_weights()[0]

In [ ]:
e.name

In [ ]:
w, b = model.get_layer("conv2d_3").get_weights()

In [ ]:
import pandas as pd

In [ ]:
pd.Series(b).plot(kind='bar')

In [ ]:
import seaborn as sns
fig, ax = plt.subplots(ncols=6)
for i in range(6):
    sns.heatmap(w[:, :, 0, i], ax=ax[i], annot=True)

In [ ]:
import numpy as np
np.argmax(model.predict(x_test[0:1]))

In [ ]:
test_sample = 5
plt.imshow(x_test[test_sample][:, :, 0], cmap=cm.get_cmap("Greys"))
pred = np.argmax(model.predict(x_test[test_sample:test_sample+1])[0])
plt.title("Predicted = {}, GT = {}".format(pred, np.argmax(y_test[test_sample])))

In [ ]:
np.argmax(model.predict(x_test[0:1])[0])

### Figure out which ones we are getting wrong

In [ ]:
pred_overall = np.argmax(model.predict(x_test), axis=1)

In [ ]:
gt_overall = np.argmax(y_test, axis=1)

In [ ]:
np.where(np.not_equal(pred_overall, gt_overall))[0]

In [ ]:
pred_overall

In [ ]:
def plot_prediction(test_sample):
    plt.imshow(x_test[test_sample][:, :, 0], cmap=cm.get_cmap("Greys"))
    pred = np.argmax(model.predict(x_test[test_sample:test_sample+1])[0])
    plt.title("Predicted = {}, GT = {}".format(pred, np.argmax(y_test[test_sample])))

In [ ]:
plot_prediction(359)

In [ ]:
plot_prediction(9729)

In [ ]:
plot_prediction(9634)

In [ ]:
### Feature map

In [ ]:
fm_model = keras.Model(inputs=model.inputs, outputs=model.layers[2].output)

In [ ]:
model.summary()

In [ ]:
fm_model.summary()

In [ ]:
fm_model.predict(x_test[test_sample:test_sample+1]).shape

In [ ]:
test_sample = 88
fm_1 = fm_model.predict(x_test[test_sample:test_sample+1])[0, :, :, :]

In [ ]:
fig, ax = plt.subplots(ncols=16, figsize=(20, 4))
for i in range(16):
    ax[i].imshow(fm_1[:, :, i], cmap="Greys")